Download classification dataset:

https://archive.ics.uci.edu/dataset/53/iris


Data definition, from iris.names:

7. Attribute Information:
   1. sepal length in cm (sepal_length)
   2. sepal width in cm (sepal_width)
   3. petal length in cm (petal_length)
   4. petal width in cm (petal_width)
   5. class: ==> class
      -- Iris Setosa
      -- Iris Versicolour
      -- Iris Virginica


In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split


In [3]:
column_names=["sepal_length", "sepal_width", "petal_length", "petal_width", "species"]
data = pd.read_csv('data/iris.data', names=column_names)

# make species a categorical column
data['species'] = data['species'].astype('category')

X = data.drop(["species"], axis=1) 
y = pd.get_dummies(data["species"])


Let's make sure the shapes are what we expected:

In [4]:
assert(X.shape == (150,4))
assert(y.shape == (150,3))

Let's look at the input data

In [5]:
X.info()
#y.value_counts

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  150 non-null    float64
 1   sepal_width   150 non-null    float64
 2   petal_length  150 non-null    float64
 3   petal_width   150 non-null    float64
dtypes: float64(4)
memory usage: 4.8 KB


In [6]:
# y now has evenly-divided categories -- one of each type
y.value_counts()

Iris-setosa  Iris-versicolor  Iris-virginica
False        False            True              50
             True             False             50
True         False            False             50
Name: count, dtype: int64

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) 

Size of inputs
20% are reserved for testing


In [8]:
print("total number of examples   ", len(X))
print("number of training examples", len(X_train))
print("number of test examples    ", len(X_test))

total number of examples    150
number of training examples 120
number of test examples     30


Sample training data


In [9]:
print("** X_train info")
X_train.info()

print("\n** X_train values[0]")
print(X_train.values[0])

print("]\n** y info")
y_train.info()


** X_train info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sepal_length  120 non-null    float64
 1   sepal_width   120 non-null    float64
 2   petal_length  120 non-null    float64
 3   petal_width   120 non-null    float64
dtypes: float64(4)
memory usage: 4.7 KB

** X_train values[0]
[4.6 3.6 1.  0.2]
]
** y info
<class 'pandas.core.frame.DataFrame'>
Index: 120 entries, 22 to 102
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Iris-setosa      120 non-null    bool 
 1   Iris-versicolor  120 non-null    bool 
 2   Iris-virginica   120 non-null    bool 
dtypes: bool(3)
memory usage: 1.3 KB


In [10]:
from micrograd.engine import Value
from micrograd.nn import Neuron, Layer, MLP

In [11]:
model = MLP(4, [16, 16, 3]) # 2-layer neural network
print(model)
print("number of parameters", len(model.parameters()))

Multi-Layer Perceptron Structure:
 Layer 1/3 - Shape of the layer is: 4 X 16 (nin X nout)
	[Neuron 0: ReLUNeuron(4) -> w0 = 0.3948, w1 =-0.2549, w2 = 0.3716, w3 =-0.5282, b = 0.0000 ... Neuron 15: ReLUNeuron(4) -> w0 = 0.6582, w1 =-0.7929, w2 =-0.5305, w3 =-0.4498, b = 0.0000]
 Layer 2/3 - Shape of the layer is: 16 X 16 (nin X nout)
	[Neuron 0: ReLUNeuron(16) -> w0 = 0.5036, w1 = 0.9152, w2 = 0.4881, w3 =-0.4654, w4 = 0.0421, w5 = 0.2164, w6 = 0.0339, w7 = 0.3066, w8 = 0.5927, w9 = 0.2476, w10 = 0.0758, w11 = 0.6314, w12 = 0.6562, w13 = 0.6808, w14 =-0.1039, w15 =-0.3901, b = 0.0000 ... Neuron 15: ReLUNeuron(16) -> w0 =-0.9588, w1 = 0.6932, w2 = 0.3963, w3 = 0.6125, w4 = 0.7177, w5 =-0.1290, w6 = 0.5321, w7 = 0.7262, w8 = 0.3856, w9 =-0.8338, w10 = 0.2332, w11 =-0.0969, w12 =-0.8459, w13 = 0.3809, w14 =-0.9910, w15 =-0.4183, b = 0.0000]
 Layer 3/3 - Shape of the layer is: 16 X 3 (nin X nout)
	[Neuron 0: LinearNeuron(16) -> w0 = 0.2685, w1 =-0.6654, w2 = 0.1269, w3 = 0.5271, w4 =-0.3850

In [12]:
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [13]:
def sparse_categorical_crossentropy(y_true, y_pred):
    """
    Calculates the sparse categorical cross-entropy loss.

    Args:
        y_true: Array of one-hot encoded true labels of shape 3xlen(y_pred).
        y_pred: Array of predicted probabilities (2D array).

    Returns:
        float: The calculated loss.
    """
    assert y_true.shape == (120, 3)
    assert y_pred.shape == (120, 3)
    
    def get_data(v):
        return v.data
    
    y_pred_values = np.array(list(map(get_data, y_pred)))
    print(f'shape: {y_pred_values.shape}')
    print(f'y_pred_values[0]: {y_pred_values[0]}\ny_true[0]:{y_true[0]}')
    # Calculate loss
    loss = -np.mean(np.sum(y_true * np.log(y_pred_values), axis=-1))
    return loss

In [14]:
l = [Value(data=0.8422718902719732),
 Value(data=-2.058144615008854),
 Value(data=1.7024704624934488)]

def get_data(v):
    return v.data
  
res = np.array(list(map(get_data, l)))
print(res)

[ 0.84227189 -2.05814462  1.70247046]


In [15]:
X_train_np = X_train.values
y_train_np = y_train.values

assert X_train_np.shape == (120, 4)
assert y_train_np.shape == (120, 3)

In [16]:


# loss function
def loss(batch_size=None):

    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X_train_np, y_train_np
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X_train_np[ri], y_train_np[ri]
    inputs = [list(map(Value, xrow)) for xrow in Xb]
    #print("Xb:", Xb[:1], "inputs:", inputs[:1])
    
    # forward the model to get scores
    scores = list(map(model, inputs))
    print("\n\n****** ", scores[0])
    print("***************************************")
    scores = np.array(scores)
    #print("yb: ", yb, " scores: ", scores)
    losses = sparse_categorical_crossentropy(yb, scores)
    data_loss = sum(losses) * (1.0 / len(losses))
    # L2 regularization
    alpha = 1e-4
    reg_loss = alpha * sum((p*p for p in model.parameters()))
    total_loss = data_loss + reg_loss
    
    # also get accuracy
    accuracy = [(yi > 0) == (scorei.data > 0) for yi, scorei in zip(yb, scores)]
    return total_loss, sum(accuracy) / len(accuracy)

total_loss, acc = loss()
print("initial total_loss: ", total_loss, " activation: ", acc)

self:  ReLUNeuron(4) -> w0 = 0.3948, w1 =-0.2549, w2 = 0.3716, w3 =-0.5282, b = 0.0000 act:  Value(data=1.1646072674307684, grad=0)
self:  ReLUNeuron(4) -> w0 =-0.4863, w1 =-0.6087, w2 =-0.3284, w3 =-0.5274, b = 0.0000 act:  Value(data=-4.861924133317682, grad=0)
self:  ReLUNeuron(4) -> w0 = 0.1277, w1 = 0.0630, w2 = 0.5444, w3 = 0.4210, b = 0.0000 act:  Value(data=1.4426900203420077, grad=0)
self:  ReLUNeuron(4) -> w0 = 0.7420, w1 =-0.5628, w2 =-0.2347, w3 =-0.4307, b = 0.0000 act:  Value(data=1.0661274528954399, grad=0)
self:  ReLUNeuron(4) -> w0 = 0.1061, w1 = 0.4911, w2 = 0.6261, w3 =-0.2883, b = 0.0000 act:  Value(data=2.824179732355309, grad=0)
self:  ReLUNeuron(4) -> w0 = 0.4811, w1 = 0.4536, w2 =-0.4865, w3 = 0.2994, b = 0.0000 act:  Value(data=3.419234383902562, grad=0)
self:  ReLUNeuron(4) -> w0 = 0.4818, w1 =-0.1428, w2 = 0.3455, w3 = 0.5374, b = 0.0000 act:  Value(data=2.154931818277622, grad=0)
self:  ReLUNeuron(4) -> w0 =-0.3324, w1 = 0.2852, w2 = 0.5922, w3 = 0.4602, b =

TypeError: loop of ufunc does not support argument 0 of type Value which has no callable log method

In [ ]:
# optimization
for k in range(100):
    
    # forward
    total_loss, acc = loss()
    
    # backward
    model.zero_grad()
    total_loss.backward()
    
    # update (sgd)
    learning_rate = 1.0 - 0.9*k/100    
    for p in model.parameters():
        p.data -= learning_rate * p.grad
    
    if k % 5 == 0:
        print(f"step {k} loss {total_loss.data}, accuracy {acc*100}%")
